# Section 7: Retrieve and analyze the results of quantum circuits

### References

- https://quantum.cloud.ibm.com/docs/en/guides/monitor-job

## Monitor Jobs

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name="ibm_fez", limit=2)

[<RuntimeJobV2('d7n2j1jaq2pc73a29j7g', 'sampler')>,
 <RuntimeJobV2('d7n2h3s3g2mc73932lk0', 'sampler')>]

In [9]:
service.job("d7o44c62jamc73bp5u70")

<RuntimeJobV2('d7o44c62jamc73bp5u70', 'estimator')>

| Method | What it does |
|---|---|
| `job.result()` | Gets job results. **Blocking call** — waits until the job completes. |
| `job.job_id()` | Returns the unique job ID. Save this if you need results later. |
| `job.status()` | Checks the current job status. |
| `job.cancel()` | Cancels a running job. |
| `service.job(<job_id>)` | Retrieves a previously submitted job using its ID. |
| `service.jobs(backend_name=...)` | Retrieves multiple jobs, optionally filtered. Default `limit` is **10**. |

## Workload Usage

- Session usage is the wall-clock time the session is active.
- Batch usage is the sum of quantum time (time spent by the QPU complex to process your job) of all jobs in the batch.
- Single job usage is the quantum time the job uses in processing.

When a job is failed or canceled, the reported usage is as follows:

- Job or batch mode: The reported usage is the time the QPU was locked for executing your workload until the time it failed or was canceled. Therefore, if the failure or cancellation occurred before the lock, the reported usage is zero. Otherwise, the workload's reported usage is the amount of usage before the workload failed or was canceled. Thus, some failed jobs do not appear in your reported usage and others do.

- Session mode: The reported usage is the wall-clock time the session is active, regardless of the number of jobs that fail or are canceled.

### Service-calculated maximum execution time
The service calculates an appropriate job timeout value based on the input circuits and options. This service-calculated timeout is capped at 3 hours to ensure fair device usage. If a max_execution_time is also specified for the job, the lesser of the two values is used.

For example, if you specify max_execution_time=5000 (approximately 83 minutes), but the service determines it should not take more than 5 minutes (300 seconds) to execute the job, then the job is canceled after 5 minutes.

### Batch maximum execution time
When a batch is started, it is assigned a maximum time to live (maximum TTL) value. After this TTL is reached, the batch is terminated, any jobs that are already running continue running, and any queued jobs that remain in the batch are put into a failed state.

Batches also have an interactive time to live (interactive TTL) value between jobs that cannot be configured. If you don't explicitly close a batch, it is deactivated after the interactive TTL expires and can be reactivated at any time until it reaches its maximum TTL.

### Session maximum execution time
When a session is started, it is assigned a maximum TTL value that determines how long a session can run. After this TTL is reached, the session is terminated, any jobs that are already running continue running, and any queued jobs that remain in the session are put into a failed state.

There is also an interactive TTL value that cannot be configured. If no session jobs are queued within that window, the session is temporarily deactivated.

### Get a count of how many gates will be run

In [12]:
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

num_qubits = 50
ghz = QuantumCircuit(num_qubits)
ghz.h(range(num_qubits))
ghz.cx(0, range(1, num_qubits))
op_counts = ghz.count_ops()
print(f"Pre-Transpilation gates: {op_counts}")

Pre-Transpilation gates: OrderedDict({'h': 50, 'cx': 49})


In [13]:
# Choose the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
transpiled_ghz = pm.run(ghz)
op_counts = transpiled_ghz.count_ops()

print(f"Post-Transpilation gates: {op_counts}")

Post-Transpilation gates: OrderedDict({'sx': 374, 'rz': 228, 'cz': 184, 'x': 10})


In [14]:
backend.name

'ibm_fez'

## Job Tags

| Action | How |
|---|---|
| Assign tags before submitting | `sampler.options.environment.job_tags = ["tag1", "tag2"]` |
| Update tags after submitting | `job.update_tags(["tag1", "tag2", "tag3"])` |
| Retrieve jobs by tag | `service.jobs(job_tags=["tag1", "tag2"])` |
| View tags on a job | `job.tags` |


In [17]:
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler

service = QiskitRuntimeService()
backend = service.least_busy(simulator=False, operational=True)

sampler = Sampler(backend)

# Assign tags before submitting
sampler.options.environment.job_tags = ["experiment-2025", "sampler-example"]

job = sampler.run([transpiled_ghz])

# View tags
print(job.tags)

# Update tags (must include original tags if you want to keep them)
job.update_tags(["experiment-2025", "sampler-example", "127-qubit"])

# Retrieve jobs by tag
service.jobs(job_tags=["experiment-2025"])

C:\Users\arpit\anaconda3\Lib\site-packages\qiskit_ibm_runtime\utils\validations.py:42: UserWarning: The 0-th circuit has no output classical registers so the result will be empty. Did you mean to add measurement instructions?
  warnings.warn(


['experiment-2025', 'sampler-example']


[<RuntimeJobV2('d7o8apak4prs73dt64ug', 'sampler')>]

In [18]:
service.jobs(job_tags=["127-qubit"])

[<RuntimeJobV2('d7o8apak4prs73dt64ug', 'sampler')>]

In [20]:
job.update_tags(["sampler-example", "127-qubit"])

['127-qubit', 'sampler-example']

## Retrieve and save job results

Quantum workflows often take a while to complete and can run over many sessions. Restarting your Python kernel means you'll lose any results stored in memory. To avoid loss of data, you can save results to file and retrieve results of past jobs from IBM Quantum® so your next session can continue where you left off.

In [21]:
import datetime
from qiskit_ibm_runtime import QiskitRuntimeService

three_months_ago = datetime.datetime.now() - datetime.timedelta(days=90)

service = QiskitRuntimeService()
jobs_in_last_three_months = service.jobs(created_after=three_months_ago)
jobs_in_last_three_months[:3]  # show first three jobs

[<RuntimeJobV2('d7o8apak4prs73dt64ug', 'sampler')>,
 <RuntimeJobV2('d7o44c62jamc73bp5u70', 'estimator')>,
 <RuntimeJobV2('d7o40hc97osc73dso9cg', 'estimator')>]

In [22]:
# Get ID of most recent successful job for demonstration.
# This will not work if you've never successfully run a job.
successful_job = next(
    j for j in service.jobs(limit=1000) if j.status() == "DONE"
)
job_id = successful_job.job_id()
print(job_id)

d7o8apak4prs73dt64ug


In [23]:
retrieved_job = service.job(job_id)
retrieved_job.result()

PrimitiveResult([SamplerPubResult(data=DataBin(), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-04-28 10:01:24', stop='2026-04-28 10:01:26', size=4096>)])}, 'version': 2})

In [24]:
import json
from qiskit_ibm_runtime import RuntimeEncoder

with open("result.json", "w") as file:
    json.dump(retrieved_job.result(), file, cls=RuntimeEncoder)

In [25]:
from qiskit_ibm_runtime import RuntimeDecoder

with open("result.json", "r") as file:
    result = json.load(file, cls=RuntimeDecoder)

result

PrimitiveResult([SamplerPubResult(data=DataBin(), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-04-28 10:01:24', stop='2026-04-28 10:01:26', size=4096>)])}, 'version': 2})

## BasePrimitiveJob

It is the abstract base class that defines the interface for the job handle object you get back from calling **primitive.run()**. Each primitives provider (like qiskit_ibm_runtime) has its own subclass with additional functionality on top of this interface. 

Each implementer of the primitives should provide a concrete implementation of this interface. There are no provided methods on the base implementation, other than the job_id() getter, since a string key uniquely identifying jobs is a requirement of all primitives.
IBM Runtime and Aer each take that template and fill in the actual implementation for their own environment.


| Method | Returns | Blocking? | Description |
|---|---|---|---|
| `job.job_id()` | `str` | No | Unique ID for the job |
| `job.result()` | `ResultT` | **Yes** | Waits and returns results |
| `job.status()` | `StatusT` | No | Current status of the job |
| `job.done()` | `bool` | No | `True` if job completed **successfully** |
| `job.running()` | `bool` | No | `True` if job is actively running |
| `job.cancelled()` | `bool` | No | `True` if job was cancelled |
| `job.in_final_state()` | `bool` | No | `True` if job is in `DONE` **or** `ERROR` state |
| `job.cancel()` | `None` | No | Attempts to cancel the job |

---

### ⚠️ Key Distinctions

| | `done()` | `in_final_state()` |
|---|---|---|
| Successful completion | ✅ True | ✅ True |
| Error/failed | ❌ False | ✅ True |
| Still running | ❌ False | ❌ False |

### ⚠️ Blocking vs Non-Blocking
- **Non-blocking** (return immediately): `status()`, `done()`, `running()`, `cancelled()`, `in_final_state()`
- **Blocking** (waits for job to finish): `result()`

## `qiskit.providers` — Providers Interface

---

### Abstract Classes

| Class | Purpose |
|---|---|
| `BackendV2` | Abstract class for all backends |
| `Options` | Base options object for backend configuration |
| `JobV1` | Abstract class for job handles |
| `JobStatus` | Enum for job status values |

---

### Exceptions

| Exception | When Raised |
|---|---|
| `JobError` | Base class for all job errors |
| `JobTimeoutError` | When a job times out *(subclass of `JobError`)* |
| `QiskitBackendNotFoundError` | When a backend lookup fails |

---

### `BackendV2` — 4 Required Properties/Methods

| Property/Method | Purpose |
|---|---|
| `target` | Describes the backend model to the compiler |
| `max_circuits` | Max number of circuits per batch job (`None` = no limit) |
| `run()` | Accepts and submits job circuits |
| `_default_options()` | Defines user-configurable options and their defaults |

```python
@classmethod
def _default_options(cls):
    return Options(shots=1024)  # example

self.options.set_validator("shots", (1, 4096))  # optional validator
```

---

### `JobStatus` Enum Values

```python
from qiskit.providers.jobstatus import JobStatus

JobStatus.INITIALIZING
JobStatus.QUEUED
JobStatus.VALIDATING
JobStatus.RUNNING
JobStatus.DONE
JobStatus.ERROR
JobStatus.CANCELLED
```

---

### Exceptions Usage Example

```python
from qiskit.providers import JobError, JobTimeoutError, QiskitBackendNotFoundError

try:
    result = job.result()
except JobTimeoutError:
    print("Job timed out")
except JobError:
    print("Job failed")
```

---

### ⚠️ Key Points

- `JobTimeoutError` is a **subclass** of `JobError`
- `BackendV2` requires ALL 4: `target`, `max_circuits`, `run()`, `_default_options()`
- Primitives (`BaseSampler`, `BaseEstimator`) are designed to allow provider-specific
  implementations with custom error mitigation, batching, and caching

## Session

In [27]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, Session, SamplerV2 as Sampler

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Bell Circuit
qr = QuantumRegister(2, name="qr")
cr = ClassicalRegister(2, name="cr")
qc = QuantumCircuit(qr, cr, name="bell")
qc.h(qr[0])
qc.cx(qr[0], qr[1])
qc.measure(qr, cr)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

with Session(backend=backend) as session:
    sampler = Sampler(mode=session)
    job = sampler.run([isa_circuit])
    pub_result = job.result()[0]
    print(f"Sampler job ID: {job.job_id()}")
    print(f"Counts: {pub_result.data.cr.get_counts()}")


> Groups iterative calls to the quantum computer. Session starts when the first job starts;
> subsequent jobs are **prioritized** by the scheduler.

---

### Constructor

```python
from qiskit_ibm_runtime import Session

Session(backend, max_time=None)
# max_time can be int (seconds) or string e.g. "2h 30m 40s"
```

---

### Attributes

| Attribute | Returns | Description |
|---|---|---|
| `session.service` | `QiskitRuntimeService` | Service associated with this session |
| `session.session_id` | `str` | Session ID (`None` if simulator) |

---

### Methods

| Method | Returns | Description |
|---|---|---|
| `session.backend()` | `str \| None` | Backend used for this session |
| `session.status()` | `str \| None` | Current session status |
| `session.details()` | `dict \| None` | Full session details dictionary |
| `session.usage()` | `float \| None` | Session usage in seconds |
| `session.close()` | `None` | Stop accepting new jobs; **existing jobs finish** |
| `session.cancel()` | `None` | Cancel **all pending jobs** immediately |
| `Session.from_id(session_id, service)` | `Session` | Reconnect to an existing session by ID |

---

### ⚠️ `close()` vs `cancel()`

| | `close()` | `cancel()` |
|---|---|---|
| Accepts new jobs? | ❌ No | ❌ No |
| Existing jobs finish? | ✅ Yes | ❌ No — all pending jobs cancelled |

---

### ⚠️ `status()` — 4 Possible Values

```python
"Pending"                           # created but not yet active
"In progress, accepting new jobs"   # active, open to new jobs
"In progress, not accepting new jobs" # active but closed to new jobs
"Closed"                            # max_time expired or explicitly closed
```

---

### ⚠️ `details()` 

```python
session.details()
# Returns dict with keys:
{
  "id":                  # session ID
  "backend_name":        # backend used
  "interactive_timeout": # max idle time (secs) between jobs before deactivation
  "max_time":            # max allowed session time (secs)
  "active_timeout":      # max time session can stay active (secs)
  "state":               # open | active | inactive | closed
  "accepting_jobs":      # bool — is session accepting new jobs?
  "last_job_started":    # timestamp
  "last_job_completed":  # timestamp ← was in sample exam answer
  "started_at":          # timestamp
  "closed_at":           # timestamp
  "activated_at":        # timestamp
  "mode":                # execution mode
  "usage_time":          # seconds the QPU was committed to this session
}
```

---

### `Session.from_id()` — Reconnect to Existing Session

```python
from qiskit_ibm_runtime import QiskitRuntimeService, Session

service = QiskitRuntimeService()
job = service.job(<job_id>)
existing_session_id = job.session_id

new_session = Session.from_id(existing_session_id, service)
```